# Engenharia de Contexto para RAG sobre Dados Tabulares — FinQA

Estudo controlado de como a **representação da tabela no prompt** afeta a **acurácia** e o **custo em tokens** de LLMs abertos de 4B a 8B (Llama 3.1 8B, Qwen3-8B, Phi-4-mini 3.8B) no FinQA, sem ajuste fino.

**Condições** (texto do relatório fixo em todas; detalhes em `docs/METODOLOGIA.md`):

| Código | Representação | Papel |
|---|---|---|
| R0 | Bruta — células separadas por espaço, uma linha por linha | controle |
| R1 | Markdown — tabela completa | referência |
| R2 | JSON — lista de objetos por linha | formato |
| R3 | Linearizada — uma frase por célula | formato |
| R4 | Seletiva — cabeçalho + *k* linhas mais relevantes, em Markdown | seleção |
| R5 | Gold-tabela — cabeçalho + linhas do gabarito, em Markdown | teto (diagnóstico) |

R0–R3 têm o mesmo conteúdo (efeito de **formato**); R1, R4 e R5 têm o mesmo formato (efeito de **seleção**).

Repositório: https://github.com/Arthur-Marques-IA/NLP_Fin-RAG

In [ ]:
# Ambiente: Python, PyTorch e GPU
import sys, os, torch
print("Python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, f"{p.total_memory/1e9:.1f} GB")
else:
    print("ATENÇÃO: sem GPU — troque o runtime para GPU antes da etapa de inferência.")

## 1. Dados — FinQA (dev e teste oficiais)

Baixamos os splits do repositório oficial ([czyssrs/FinQA](https://github.com/czyssrs/FinQA)). Cada exemplo traz `pre_text`, `table`, `post_text` e o bloco `qa` com pergunta, programa anotado (`program`), resposta executada (`exe_ans`) e evidências do gabarito (`gold_inds`).

| Split | Uso neste estudo |
|---|---|
| train | só para sortear os exemplos few-shot (notebook 02) |
| **dev** | escolher *k* da seletiva e pilotar prompt/parser |
| **test** | execução final, uma vez, com tudo congelado |

In [ ]:
# Download do FinQA (dev e teste)
import json, pathlib, urllib.request

DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)
FINQA_BASE = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/"

def load_finqa(split):
    path = DATA_DIR / f"finqa_{split}.json"
    if not path.exists():
        urllib.request.urlretrieve(FINQA_BASE + f"{split}.json", path)
    with open(path, encoding="utf-8") as f:
        return json.load(f)

finqa_dev = load_finqa("dev")
finqa_test = load_finqa("test")
print("Perguntas no dev:  ", len(finqa_dev))
print("Perguntas no teste:", len(finqa_test))
print("IDs em comum dev/teste:", len({e["id"] for e in finqa_dev} & {e["id"] for e in finqa_test}))
print("Chaves de um exemplo:", list(finqa_test[0].keys()))
print("Chaves de qa:", list(finqa_test[0]["qa"].keys()))

In [ ]:
# Inspeção de um exemplo
ex = finqa_test[0]
qa = ex["qa"]
print("ID:", ex["id"])
print("Pergunta:", qa["question"])
print("Programa:", qa["program"])
print("Resposta (exe_ans):", qa["exe_ans"], "| answer:", qa["answer"])
print("Linhas de tabela no gabarito (ann_table_rows):", qa["ann_table_rows"])
print("Linhas de texto no gabarito (ann_text_rows):", qa["ann_text_rows"])
print("gold_inds:", qa["gold_inds"])
print(f"\nTabela ({len(ex['table'])} linhas):")
for row in ex["table"]:
    print("  ", row)
print(f"\npre_text: {len(ex['pre_text'])} frases | post_text: {len(ex['post_text'])} frases")

## 2. Metadados para a análise

Para cada pergunta extraímos as variáveis de corte do estudo:
- **`n_steps`** — número de operações no programa anotado; corte principal da H3 é **1 × 2+** (`steps_main`), e 1 / 2 / 3+ (`steps_bin`) é exploratório, porque 3+ tem só 84 perguntas;
- **`evidence`** — origem da evidência no gabarito: `table`, `text` ou `both` (a partir das chaves de `gold_inds`);
- **`n_rows` / `n_cols`** — tamanho da tabela (linhas de dados, sem o cabeçalho).

> Os campos `table_retrieved*`, `text_retrieved*`, `tfidftopn` e `model_input` vêm de recuperadores do artigo original e **não** são usados nos prompts.

In [ ]:
# Metadados por pergunta
import re
import pandas as pd

OP_RE = re.compile(r"[a-z_]+\(")

def count_steps(program: str) -> int:
    return len(OP_RE.findall(program))

def evidence_source(gold_inds: dict) -> str:
    kinds = {k.split("_")[0] for k in gold_inds}
    if kinds == {"table"}:
        return "table"
    if kinds == {"text"}:
        return "text"
    return "both"

def n_data_rows(table):
    # tabela de uma linha só: essa linha é dado (ver normalize_table na seção 3)
    return len(table) - 1 if len(table) > 1 else len(table)

meta = pd.DataFrame([{
    "id": ex["id"],
    "n_steps": count_steps(ex["qa"]["program"]),
    "evidence": evidence_source(ex["qa"]["gold_inds"]),
    "n_rows": n_data_rows(ex["table"]),
    "n_cols": max(len(r) for r in ex["table"]),
    "gold_answer": ex["qa"]["exe_ans"],
} for ex in finqa_test])
meta["steps_bin"] = meta["n_steps"].clip(upper=3).map({1: "1", 2: "2", 3: "3+"})
meta["steps_main"] = meta["n_steps"].map(lambda s: "1" if s == 1 else "2+")

print("Passos:\n", meta["steps_bin"].value_counts().sort_index().to_string(), "\n")
print("Passos (corte principal da H3):\n", meta["steps_main"].value_counts().sort_index().to_string(), "\n")
print("Origem da evidência:\n", meta["evidence"].value_counts().to_string(), "\n")
print("Linhas de dados por tabela:\n", meta["n_rows"].describe().round(2).to_string())
print("\nTipos de resposta:", meta["gold_answer"].map(lambda v: type(v).__name__).value_counts().to_dict())

## 3. Representações da tabela

Todas partem da mesma `table` do FinQA (linha 0 = cabeçalho, coluna 0 = rótulo da linha) e mantêm os valores das células **sem alteração**, para que só a forma mude.

- **R0 Bruta:** células não vazias separadas por espaço, uma linha de texto por linha da tabela, sem marcação de coluna — como a tabela sai de um extrator de PDF.
- **R1 Markdown / R2 JSON:** a primeira coluna do cabeçalho, quase sempre vazia, recebe o nome `item` (colunas repetidas ganham sufixo `_1`, `_2`); é a única palavra adicionada.
- **R3 Linearizada:** o template do próprio FinQA, `the <linha> of <coluna> is <valor> ;`, uma frase por célula; linhas só com rótulo viram `<rótulo> ;` para não perder conteúdo.
- **R4 Seletiva:** pontua cada linha pela sobreposição lexical com a pergunta (termos sem stopwords; números valem 2, palavras 1) e mantém as `k` melhores, na ordem original. Usa só `(tabela, pergunta)`; não usa gabarito nem LLM (custo zero em tokens de modelo). **`k` é escolhido no dev** (seção 3.1).
- **R5 Gold-tabela:** cabeçalho + linhas das chaves `table_i` de `gold_inds` (o campo `ann_table_rows` vem vazio em ~110 perguntas). É o teto da seleção **de tabela**: não fornece os fatos do texto.

In [ ]:
# Serializações da tabela (R0–R5)
def _header_names(header):
    """Nomes de coluna únicos; a primeira coluna vazia vira 'item'."""
    names, seen = [], {}
    for i, h in enumerate(header):
        name = h.strip() or ("item" if i == 0 else f"col_{i}")
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        names.append(name)
    return names

def _pad(row, n):
    return list(row) + [""] * (n - len(row))

def _cell(c):
    return str(c).strip()

def normalize_table(table):
    """Tabela de uma linha só (4 perguntas do teste): essa linha é dado, com cabeçalho vazio.
    Sem isso, R2 e R3 sairiam vazias e perderiam todo o conteúdo."""
    if len(table) == 1:
        return [[""] * len(table[0]), table[0]]
    return table

def to_raw(table):
    """R0: células não vazias separadas por espaço, uma linha por linha da tabela."""
    lines = [" ".join(_cell(c) for c in row if _cell(c)) for row in table]
    return "\n".join(l for l in lines if l)

def to_markdown(table):
    """R1 (e base de R4/R5)."""
    if not table:
        return ""
    n = max(len(r) for r in table)
    header = _header_names(_pad(table[0], n))
    esc = lambda c: _cell(c).replace("|", "\\|")
    lines = ["| " + " | ".join(map(esc, header)) + " |",
             "|" + "|".join(["---"] * n) + "|"]
    lines += ["| " + " | ".join(map(esc, _pad(r, n))) + " |" for r in table[1:]]
    return "\n".join(lines)

def to_json(table):
    """R2: lista de objetos, um por linha de dados."""
    if not table:
        return "[]"
    n = max(len(r) for r in table)
    header = _header_names(_pad(table[0], n))
    rows = [dict(zip(header, map(_cell, _pad(r, n)))) for r in table[1:]]
    return json.dumps(rows, ensure_ascii=False)

def linearize_row(header, row):
    """Frases de uma linha no template do FinQA, reproduzido à risca:
    '[<cabeçalho da coluna 0>] the <linha> of <coluna> is <valor> ; ...'
    (coluna sem nome: 'the <linha> of is <valor> ;', como no FinQA).
    Linha só com rótulo (ex.: título de seção) vira '<rótulo> ;'. Valores nunca são alterados."""
    n = max(len(header), len(row))
    header, row = _pad(header, n), _pad(row, n)
    label = _cell(row[0])
    sents = []
    for col, val in zip(header[1:], row[1:]):
        if _cell(val):
            of_col = f"of {_cell(col)}" if _cell(col) else "of"
            sents.append(f"the {label} {of_col} is {_cell(val)} ;")
    if not sents and label:
        sents = [f"{label} ;"]
    if _cell(header[0]) and sents:
        sents[0] = f"{_cell(header[0])} {sents[0]}"
    return sents

def to_linearized(table):
    """R3: uma frase por célula."""
    if not table:
        return ""
    return "\n".join(s for r in table[1:] for s in linearize_row(table[0], r))

# --- Gabarito de linhas (R5) ---
def gold_table_rows(ex):
    """Linhas de tabela do gabarito, a partir das chaves 'table_i' de gold_inds.
    (ann_table_rows vem vazio em ~110 perguntas do teste; gold_inds é mais completo.)"""
    return sorted(int(k.split("_")[1]) for k in ex["qa"]["gold_inds"] if k.startswith("table_"))

# --- Seleção de linhas (R4), sem gabarito ---
STOPWORDS = set("""a an the of in on for to by and or is are was were be been what which how
much many did does do from with as at that this these those its it their than then
percent percentage change total amount value values ratio year years""".split())
TOKEN_RE = re.compile(r"[a-z]+|\d+(?:\.\d+)?")

def _terms(text):
    return [t for t in TOKEN_RE.findall(str(text).lower()) if t not in STOPWORDS]

def select_rows(table, question, k):
    """Índices (>=1) das k linhas com maior sobreposição com a pergunta, na ordem original.
    Empate: a linha que vem antes. Recebe SÓ a tabela e a pergunta."""
    q = set(_terms(question))
    scored = []
    for i, row in enumerate(table[1:], start=1):
        terms = set(_terms(" ".join(map(str, row))))
        score = sum(2.0 if t.isdigit() else 1.0 for t in q & terms)
        scored.append((score, i))
    top = sorted(scored, key=lambda s: (-s[0], s[1]))[:k]
    return sorted(i for _, i in top)

def subtable(table, row_ids):
    """Cabeçalho (linha 0) sempre incluído + linhas pedidas, na ordem original."""
    return [table[0]] + [table[i] for i in sorted(set(row_ids)) if 0 < i < len(table)]

REPRESENTATIONS = ["raw", "markdown", "json", "linearized", "selective", "gold"]
REP_CODE = dict(zip(REPRESENTATIONS, ["R0", "R1", "R2", "R3", "R4", "R5"]))

def serialize(ex, rep, k):
    t = normalize_table(ex["table"])
    if rep == "raw":        return to_raw(t)
    if rep == "markdown":   return to_markdown(t)
    if rep == "json":       return to_json(t)
    if rep == "linearized": return to_linearized(t)
    if rep == "selective":  return to_markdown(subtable(t, select_rows(t, ex["qa"]["question"], k)))
    if rep == "gold":
        rows = gold_table_rows(ex)
        if len(ex["table"]) == 1:          # linha 0 original virou a linha 1
            rows = [i + 1 for i in rows]
        return to_markdown(subtable(t, rows))
    raise ValueError(rep)

def build_context(ex, rep, k):
    """Texto do relatório fixo; só o bloco da tabela muda entre representações."""
    pre = " ".join(ex["pre_text"])
    post = " ".join(ex["post_text"])
    return f"{pre}\n\nTable:\n{serialize(ex, rep, k)}\n\n{post}".strip()

print("Representações:", ", ".join(f"{REP_CODE[r]}={r}" for r in REPRESENTATIONS))

### 3.1 Escolha de *k* (no dev) e subconjunto efetivo (no teste)

**Regra, fixada antes de olhar o dev:** entre *k* ∈ {1, …, 5}, usar o **menor *k*** em que a seletiva preserva **todas** as linhas do gabarito em **≥ 80%** das perguntas do dev que têm linhas de dados no gabarito; se nenhum atingir, *k* = 5.

No teste, só descrevemos o *k* escolhido. O **subconjunto efetivo** da H2 são as perguntas com tabela de mais de *k* linhas de dados **e** com linhas no gabarito: fora dele, R4 é idêntica a R1 ou não há linha a preservar.

In [ ]:
# Escolha de k no dev (regra da seção 3.1) e descrição no teste
K_CANDIDATES = (1, 2, 3, 4, 5)
K_THRESHOLD = 0.80

def gold_data_rows(ex):
    return {i for i in gold_table_rows(ex) if i > 0}   # linha 0 (cabeçalho) sempre entra

def gold_preserved(ex, k):
    """True/False se a seleção com k mantém todas as linhas do gabarito; None se não há linha a preservar."""
    gold = gold_data_rows(ex)
    if not gold:
        return None
    return gold <= set(select_rows(ex["table"], ex["qa"]["question"], k))

print("DEV — perguntas com linhas de dados no gabarito:",
      sum(1 for ex in finqa_dev if gold_data_rows(ex)), "de", len(finqa_dev))
dev_rates = {}
for k in K_CANDIDATES:
    flags = [p for p in (gold_preserved(ex, k) for ex in finqa_dev) if p is not None]
    dev_rates[k] = sum(flags) / len(flags)
    print(f"  k={k}: todas as linhas gold preservadas em {dev_rates[k]:.1%}")
K = next((k for k in K_CANDIDATES if dev_rates[k] >= K_THRESHOLD), K_CANDIDATES[-1])
print(f"\n>>> k escolhido no dev: K = {K}")

# --- Teste: só descrição com o K congelado ---
meta["gold_preserved"] = [gold_preserved(ex, K) for ex in finqa_test]
meta["has_gold_rows"] = meta["gold_preserved"].notna()
meta["effective_h2"] = meta["has_gold_rows"] & (meta["n_rows"] > K)
same = (meta["n_rows"] <= K).sum()
print(f"\nTESTE (K={K}):")
print(f"  R4 idêntica a R1 (tabela com até {K} linhas): {same} ({same/len(meta):.1%})")
print(f"  subconjunto efetivo da H2: {meta['effective_h2'].sum()} perguntas")
eff = meta[meta["effective_h2"]]
print(f"  no efetivo, seleção preservou todas as linhas gold em {eff['gold_preserved'].astype(bool).mean():.1%}"
      f" ({eff['gold_preserved'].astype(bool).sum()} preservadas / {(~eff['gold_preserved'].astype(bool)).sum()} cortadas)")

print("\nExemplo — a mesma tabela nas seis representações:\n")
for rep in REPRESENTATIONS:
    print(f"===== {REP_CODE[rep]} {rep} =====")
    print(serialize(finqa_test[0], rep, K), "\n")

## 4. Avaliação — acurácia estrita e tolerante

Gabarito: `exe_ans` (resultado da execução do programa anotado), como na acurácia de execução do FinQA.

**Extração:** lê-se **só o texto após o último `Answer:`** e o primeiro número ali (tratando `$`, `,`, `%`, `.5` e negativos entre parênteses). Sem `Answer:` ou sem número legível → **falha de formato** (conta como erro).

- **Estrita (principal):** erro relativo ≤ 1% com `pred ≈ gold` ou `pred ≈ gold × 100` (porcentagem escrita como número). A escala `÷100` **não** é aceita.
- **Tolerante (sensibilidade):** estrita **ou** o gabarito, arredondado com as casas decimais da resposta, é igual à resposta — com mesmo sinal, resposta ≠ 0 e **erro relativo ≤ 5%**.
- **Sim/não** (20 perguntas): `yes`/`no` após `Answer:`.

Validação: aplicamos o avaliador ao campo `answer` dos anotadores (prefixado com `Answer:`); as divergências restantes são ruído de anotação.

In [ ]:
# Extração e avaliação de respostas
import math

ANSWER_RE = re.compile(r"answer\s*[:：]", re.I)
NUM_RE = re.compile(r"\(?\s*-?\s*\$?\s*(?:\d[\d,]*(?:\.\d+)?|\.\d+)\s*%?\s*\)?")
REL_TOL_STRICT = 0.01
REL_TOL_LENIENT = 0.05

def answer_segment(text):
    """Texto após o ÚLTIMO 'Answer:'; None se não houver."""
    if text is None:
        return None
    ms = list(ANSWER_RE.finditer(str(text)))
    return str(text)[ms[-1].end():] if ms else None

def parse_number(segment):
    """(valor, casas_decimais) do primeiro número do segmento; None se não houver."""
    if segment is None:
        return None
    m = NUM_RE.search(segment)
    if not m:
        return None
    tok = m.group(0).strip()
    neg = (tok.startswith("(") and tok.endswith(")")) or tok.lstrip("( ").startswith("-")
    digits = re.sub(r"[^\d.]", "", tok)
    decimals = len(digits.split(".")[1]) if "." in digits else 0
    val = float(digits)
    return (-val if neg else val), decimals

def parse_yes_no(segment):
    if segment is None:
        return None
    m = re.search(r"\b(yes|no)\b", segment.lower())
    return m.group(1) if m else None

def _rel_err(pred, g):
    return abs(pred - g) / abs(g) if g != 0 else (0.0 if pred == 0 else math.inf)

def evaluate(pred_text, gold):
    """Retorna dict(format_ok, strict, lenient, pred)."""
    seg = answer_segment(pred_text)
    if isinstance(gold, str):                       # sim/não
        yn = parse_yes_no(seg)
        ok = yn == gold.strip().lower()
        return {"format_ok": yn is not None, "strict": ok, "lenient": ok, "pred": yn}
    p = parse_number(seg)
    if p is None:
        return {"format_ok": False, "strict": False, "lenient": False, "pred": None}
    pred, dec = p
    scales = (gold, gold * 100)                    # x1 e porcentagem escrita como número
    strict = any(_rel_err(pred, g) <= REL_TOL_STRICT for g in scales)
    lenient = strict or any(
        pred != 0 and (pred > 0) == (g > 0)
        and math.isclose(round(g, dec), pred, abs_tol=1e-9)
        and _rel_err(pred, g) <= REL_TOL_LENIENT
        for g in scales)
    return {"format_ok": True, "strict": strict, "lenient": lenient, "pred": pred}

# Casos da tabela 6.1 de docs/METODOLOGIA.md + extras: (saída, gabarito, estrita, tolerante, formato_ok)
PARSER_TESTS = [
    ("Answer: 94.00", 94.0, True, True, True),
    ("Answer: 14.46%", 0.14464, True, True, True),
    ("Answer: 14.46", 0.14464, True, True, True),
    ("Answer: 14%", 0.14464, False, True, True),
    ("Answer: (32.00)", -32.0, True, True, True),
    ("Answer: -32.00", -32.0, True, True, True),
    ("Answer: 1", 1.4, False, False, True),
    ("Answer: 10", 9.6, False, True, True),
    ("Answer: 0.94", 94.0, False, False, True),
    ("Answer: 0", 0.4, False, False, True),
    ("Answer: 0.35%", -0.00357, False, False, True),
    ("Answer: $1,234.50", 1234.5, True, True, True),
    ("Answer: .385", 0.38537, True, True, True),
    ("In 2015 revenue grew.\nAnswer: 94.00", 94.0, True, True, True),
    ("answer: 5.00\nWait, recheck.\nAnswer: 94.00", 94.0, True, True, True),
    ("There was no change. Answer: yes", "yes", True, True, True),
    ("Answer: no", "yes", False, False, True),
    ("The answer is 94", 94.0, False, False, False),
    ("Answer: cannot be determined", 94.0, False, False, False),
]
fails = []
for text, gold, e_s, e_l, e_f in PARSER_TESTS:
    r = evaluate(text, gold)
    good = (r["strict"], r["lenient"], r["format_ok"]) == (e_s, e_l, e_f)
    print(("OK  " if good else "ERRO"), repr(text)[:44].ljust(46), f"gold={gold!s:<8}",
          f"estrita={r['strict']!s:<5} tolerante={r['lenient']!s:<5} formato={r['format_ok']}")
    if not good:
        fails.append(text)
assert not fails, fails

# Validação: campo 'answer' dos anotadores contra exe_ans
ann = [evaluate("Answer: " + str(ex["qa"]["answer"]), ex["qa"]["exe_ans"]) for ex in finqa_test]
n = len(ann)
print(f"\n'answer' dos anotadores aceito: estrita {sum(r['strict'] for r in ann)/n:.1%} | "
      f"tolerante {sum(r['lenient'] for r in ann)/n:.1%} | sem número {sum(not r['format_ok'] for r in ann)}")

## 5. Verificações obrigatórias (seção 8 de `docs/METODOLOGIA.md`)

Garantem que as condições comparadas são o que dizemos que são. Rodam no dev **e** no teste; qualquer falha interrompe o notebook.

| # | Verificação |
|---|---|
| V1 | **Mesmo conteúdo em R0–R3:** todo valor de célula, rótulo de linha e nome de coluna com dados aparece nas quatro |
| V1b | **R3 reproduz o template do FinQA:** as frases geradas para as linhas do gabarito batem com as de `gold_inds` (ignorando espaços) |
| V2 | **Mesmo formato em R1, R4, R5:** R4 e R5 têm o cabeçalho de R1 e suas linhas são uma subsequência das linhas de R1 |
| V3 | **Sem vazamento na seleção:** `select_rows` recebe só `(table, question, k)`, e a seletiva não muda quando o gabarito é removido do exemplo |
| V4 | **Texto fixo:** fora do bloco da tabela, o contexto é idêntico nas seis condições |
| V6 | **Parser:** todos os casos de teste passam |

V5 (few-shot coerente) e V7 (piloto no dev) entram no notebook 02, junto com o prompt.

In [ ]:
# Verificações V1–V4 e V6
import inspect

ALL = [("dev", e) for e in finqa_dev] + [("test", e) for e in finqa_test]

def content_items(table):
    """Valores, rótulos de linha, nomes de coluna com dados e o cabeçalho da coluna 0:
    o que toda representação COMPLETA (R0–R3) deve conter."""
    table = normalize_table(table)
    if not table:
        return set()
    n = max(len(r) for r in table)
    header = _pad(table[0], n)
    items, cols_with_data = set(), set()
    for r in table[1:]:
        r = _pad(r, n)
        if _cell(r[0]):
            items.add(_cell(r[0]))
        for j in range(1, n):
            if _cell(r[j]):
                items.add(_cell(r[j]))
                cols_with_data.add(j)
    items |= {_cell(header[j]) for j in cols_with_data if _cell(header[j])}
    if _cell(header[0]) and len(table) > 1:
        items.add(_cell(header[0]))
    return items

def json_strings(s):
    out = set()
    for row in json.loads(s):
        out |= set(row.keys()) | set(row.values())
    return out

# V1 — mesmo conteúdo em R0–R3
v1_fail = []
for split, ex in ALL:
    items = content_items(ex["table"])
    texts = {
        "raw": serialize(ex, "raw", K),
        "markdown": serialize(ex, "markdown", K).replace("\\|", "|"),
        "linearized": serialize(ex, "linearized", K),
    }
    jvals = json_strings(serialize(ex, "json", K))
    for it in items:
        for rep, txt in texts.items():
            if it not in txt:
                v1_fail.append((split, ex["id"], rep, it))
        if not any(it == v or it in v for v in jvals):
            v1_fail.append((split, ex["id"], "json", it))
    for rep in ("raw", "markdown", "json", "linearized"):     # nenhuma representação completa vazia
        if not serialize(ex, rep, K).strip("[] \n") and items:
            v1_fail.append((split, ex["id"], rep, "<vazia>"))
print(f"V1  conteúdo igual em R0–R3: {len(v1_fail)} falhas em {len(ALL)} tabelas")
assert not v1_fail, v1_fail[:5]

# V1b — R3 reproduz o template do FinQA (gold_inds), ignorando espaços
squash = lambda s: re.sub(r"\s+", "", s)
tot = hit = 0
for split, ex in ALL:
    for key, txt in ex["qa"]["gold_inds"].items():
        if key.startswith("table_") and int(key.split("_")[1]) > 0:
            i = int(key.split("_")[1])
            tot += 1
            hit += squash(" ".join(linearize_row(ex["table"][0], ex["table"][i]))) == squash(txt)
print(f"V1b R3 igual ao template do FinQA: {hit}/{tot} ({hit/tot:.1%}) linhas do gabarito")
assert hit / tot >= 0.99

# V2 — R1, R4, R5 no mesmo formato
v2_fail = []
for split, ex in ALL:
    full = serialize(ex, "markdown", K).split("\n")
    for rep in ("selective", "gold"):
        sub = serialize(ex, rep, K).split("\n")
        it = iter(full[2:])
        if sub[:2] != full[:2] or not all(line in it for line in sub[2:]):
            v2_fail.append((split, ex["id"], rep))
print(f"V2  R4/R5 = subtabela de R1 no mesmo formato: {len(v2_fail)} falhas")
assert not v2_fail, v2_fail[:5]

# V3 — sem vazamento na seleção
assert list(inspect.signature(select_rows).parameters) == ["table", "question", "k"]
v3_fail = [ex["id"] for _, ex in ALL
           if serialize({"table": ex["table"], "qa": {"question": ex["qa"]["question"]}}, "selective", K)
           != serialize(ex, "selective", K)]
print(f"V3  seletiva independente do gabarito: {len(v3_fail)} falhas")
assert not v3_fail, v3_fail[:5]

# V4 — texto fixo fora do bloco da tabela
v4_fail = []
for split, ex in ALL:
    outside = set()
    for rep in REPRESENTATIONS:
        ctx, tab = build_context(ex, rep, K), serialize(ex, rep, K)
        head, sep, tail = ctx.partition("\nTable:\n")
        assert sep and tail.startswith(tab)
        outside.add((head, tail[len(tab):]))
    if len(outside) != 1:
        v4_fail.append((split, ex["id"]))
print(f"V4  texto idêntico fora da tabela: {len(v4_fail)} falhas")
assert not v4_fail, v4_fail[:5]

# V6 — parser
assert all((evaluate(t, g)["strict"], evaluate(t, g)["lenient"], evaluate(t, g)["format_ok"]) == (s, l, f)
           for t, g, s, l, f in PARSER_TESTS)
print(f"V6  parser: {len(PARSER_TESTS)} casos OK")
print("\nTodas as verificações passaram.")

## 6. Custo — tokens de contexto por representação

Contamos os tokens do **contexto da pergunta** (texto do relatório + tabela serializada) com o **tokenizador de cada modelo**, nas seis condições e com o *k* escolhido no dev. Tokens totais de entrada (com instrução e few-shot) e tokens de saída entram no notebook 02, com o prompt.

O Llama 3.1 é *gated* no Hugging Face: adicione um token com acesso ao modelo em **Secrets** do Colab com o nome `HF_TOKEN` (sem isso ele é pulado nesta etapa). O custo da seleção (R4) é zero em tokens de LLM, porque a seleção é lexical.

In [ ]:
# Tokenizadores dos três modelos
from transformers import AutoTokenizer

MODELS = {
    "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
    "qwen3-8b":    "Qwen/Qwen3-8B",
    "phi4-mini":   "microsoft/Phi-4-mini-instruct",
}

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

tokenizers = {}
for name, repo in MODELS.items():
    try:
        tokenizers[name] = AutoTokenizer.from_pretrained(repo, token=HF_TOKEN)
        print(f"OK    {name:12s} ({repo})")
    except Exception as e:
        print(f"PULADO {name:12s} ({repo}): {type(e).__name__} — {str(e).splitlines()[0][:110]}")

In [ ]:
# Contagem de tokens do contexto, por modelo e representação (teste, K congelado)
records = []
for ex, m in zip(finqa_test, meta.itertuples()):
    for rep in REPRESENTATIONS:
        table_txt = serialize(ex, rep, K)
        ctx_txt = build_context(ex, rep, K)
        for name, tok in tokenizers.items():
            records.append({
                "id": ex["id"], "model": name, "rep": rep, "code": REP_CODE[rep], "k": K,
                "table_tokens": len(tok.encode(table_txt, add_special_tokens=False)),
                "context_tokens": len(tok.encode(ctx_txt, add_special_tokens=False)),
                "n_rows": m.n_rows, "steps_main": m.steps_main, "evidence": m.evidence,
                "effective_h2": m.effective_h2,
            })
tokens_df = pd.DataFrame(records)
tokens_df.to_csv(DATA_DIR / "tokens_by_rep.csv", index=False)

order = [REP_CODE[r] + " " + r for r in REPRESENTATIONS]
tokens_df["cond"] = tokens_df["code"] + " " + tokens_df["rep"]
summary = (tokens_df.groupby(["model", "cond"])[["table_tokens", "context_tokens"]]
           .mean().round(1).unstack("model").reindex(order))
print(f"Média de tokens por pergunta (K={K}):\n")
print(summary.to_string())

base = tokens_df[tokens_df.rep == "markdown"].set_index(["id", "model"])["context_tokens"]
print("\nContexto relativo a R1 (média das razões por pergunta):")
for rep in REPRESENTATIONS:
    r = tokens_df[tokens_df.rep == rep].set_index(["id", "model"])["context_tokens"] / base
    eff = tokens_df[(tokens_df.rep == rep) & tokens_df.effective_h2].set_index(["id", "model"])["context_tokens"]
    r_eff = eff / base.loc[eff.index]
    print(f"  {REP_CODE[rep]} {rep:11s} todas: {r.mean():.3f} | subconjunto efetivo H2: {r_eff.mean():.3f}")

## Próximos passos (notebook 02)

1. **Prompt e few-shot** — template único (seção 3 de `docs/METODOLOGIA.md`), 3 exemplos fixos do **treino**, com as tabelas dos exemplos **na mesma representação da condição** (V5).
2. **Piloto no dev** (~150 perguntas) — falha de formato < 5% por modelo com R1 (V7); ajustes de prompt só no dev, depois congelar.
3. **Inferência no teste** — 1.147 × 6 condições × 3 modelos; gulosa, `max_new_tokens=512`, Qwen3 com `enable_thinking=False`, **mesma precisão para os três modelos**; JSONL no Drive com retomada.
4. **Análise** — contrastes pareados contra R1 (bootstrap + McNemar + Holm), H2 no subconjunto efetivo (preservadas × cortadas), cortes 1 × 2+ passos e por origem da evidência, placebo nas perguntas só de texto, fronteira de Pareto acurácia × tokens.